# Phase 1 — data preparation & split

SOFA-forecasting project. This notebook is the executable record of Phase 1; it
**orchestrates** the reusable modules in `modeling/data_prep/` and renders the
quality-control narrative. See `SOFA_forecast_project_plan.docx` for the problem
statement, decision log and roadmap.

**Task recap.** At each decision time `T ∈ {24, 30, 36, 42} h` we will forecast the
trailing-24 h SOFA over `[T−18, T+6)` from `icu_records[0, T)` only. Six-axis
(Approach 2); cardiovascular capped 0/1 (no vasopressor data). `outcomes.csv` is
never a feature — evaluation reference only.

**Phase 1 outputs** (`modeling/outputs/`): `splits.csv`, `records_clean.joblib`,
`targets.csv`, `features.csv`, `qc_report.md`.


In [1]:
import os, sys, time, pathlib
sys.path.insert(0, os.path.abspath(".."))          # project root on path

import numpy as np, pandas as pd
from scipy import stats
import joblib

from modeling import config
from modeling.data_prep import parsing, splits, targets, features, qc

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 50)
print("horizons:", config.HORIZONS_H)
for T in config.HORIZONS_H:
    print(f"  T={T:>2}h  features t<{config.feature_cutoff_min(T)}min  "
          f"target window {tuple(x//60 for x in config.target_window_min(T))}h  "
          f"now window {tuple(x//60 for x in config.now_window_min(T))}h")


C:\Conda\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


horizons: (24, 30, 36, 42)
  T=24h  features t<1440min  target window (6, 30)h  now window (0, 24)h
  T=30h  features t<1800min  target window (12, 36)h  now window (6, 30)h
  T=36h  features t<2160min  target window (18, 42)h  now window (12, 36)h
  T=42h  features t<2520min  target window (24, 48)h  now window (18, 42)h


## 1b  Train / validation / test split

Record-level (a stay's four horizon-rows never straddle splits), stratified on a
coarse SOFA-severity band × in-hospital death (`SOFA == -1` is its own band),
70 / 15 / 15, `random_state = 42`, **frozen**. No patient linkage exists, so
record-level is the finest grouping available.


In [2]:
sp = splits.make_splits()
sp.to_csv(config.SPLITS_CSV, index=False)
print(sp.split.value_counts().to_dict())
splits.split_summary(sp)


{'train': 8400, 'val': 1800, 'test': 1800}


,n,pct,sofa_mean,sofa_p25,sofa_p75,death_rate,sofa_missing,icu1,icu2,icu3,icu4
train,8400.0,70.0,6.66,3.0,9.0,0.142,282.0,0.146,0.211,0.358,0.285
val,1800.0,15.0,6.66,3.0,9.0,0.143,61.0,0.155,0.210,0.353,0.282
test,1800.0,15.0,6.67,3.0,9.0,0.142,60.0,0.148,0.210,0.362,0.280


The three splits match on SOFA mean/quartiles, death rate and ICU-type mix —
stratification is doing its job.


## 1c  Parse & clean the records

`parsing.parse_record` is lifted verbatim (rule-wise) from `sofa_calculation.ipynb`:
drop the `-1` sentinel, empty-`Parameter` rows, `MAP/NIMAP/PaO2/SaO2 ≤ 0`,
`Weight ≤ 20`, `Height < 100 or > 250`. The audit below must reproduce the
notebook's section-2 counts exactly.


In [3]:
t0 = time.time()
if config.RECORDS_CLEAN.exists():
    records = joblib.load(config.RECORDS_CLEAN)
    print(f"loaded {len(records)} cleaned records from cache in {time.time()-t0:.1f}s")
else:
    records = parsing.parse_all()
    joblib.dump(records, config.RECORDS_CLEAN, compress=3)
    print(f"parsed + cached {len(records)} records in {time.time()-t0:.1f}s")

audit = qc.audit_raw()
bad = {k: (audit['counts'][k], v) for k, v in qc.DOC_REF.items() if audit['counts'][k] != v}
print("section-7 audit mismatches vs doc:", bad or "NONE — all match")
pd.DataFrame({"computed": audit['counts'], "doc": qc.DOC_REF}).assign(
    ok=lambda d: d.computed == d.doc)


loaded 12000 cleaned records from cache in 16.3s
section-7 audit mismatches vs doc: NONE — all match


,computed,doc,ok
empty_param_rows,2403,2403,True
MAP_eq_0,86,86,True
PaO2_eq_0,3,3,True
SaO2_eq_0,2,2,True
Weight_neg1,1001,1001,True
Weight_le20,353,353,True
Weight_gt300,1,1,True
Height_lt100,17,17,True
Height_gt250,11,11,True
MechVent_rows,91644,91644,True


In [4]:
lt = audit['last_t_hours']
print(f"last observation (h): min {lt.min():.1f}  p5 {np.percentile(lt,5):.1f}  "
      f"median {np.percentile(lt,50):.1f}  max {lt.max():.1f}")
print("records ending before 42h:", int((lt < 42).sum()), " before 36h:", int((lt < 36).sum()))
qc.window_coverage(records)


last observation (h): min 0.0  p5 46.3  median 47.5  max 48.0
records ending before 42h: 171  before 36h: 83


,origin_h,window,GCS,HR,Creatinine,Platelets,Bilirubin,PaO2,FiO2,MAP|NIMAP
0,24,"[6,30)h",98.3,98.4,96.6,95.6,27.5,66.2,61.5,98.4
1,30,"[12,36)h",98.3,98.4,95.6,94.8,24.7,62.4,58.6,98.3
2,36,"[18,42)h",98.2,98.4,95.0,94.2,22.6,59.0,55.5,98.3
3,42,"[24,48)h",98.1,98.2,96.3,95.5,21.7,56.5,52.1,98.2


Every target window still has GCS / creatinine / platelets / MAP in 94–98 % of
records and *some* vital in ~99.7 %. All four horizons are viable.


## 1d  Forecast targets

For each `(RecordID, T)`: the six rule sub-scores + total on `[T−18, T+6)`
(`sofa_total`), the same on `[T−24, T)` (`sofa_now`), the per-axis and total
**delta**, the `deteriorate_24h` flag (`sofa_delta ≥ 2`), and `cardio_instability`
(min MAP < 65 in the window). Empty target windows → NaN scores, kept for accounting.


In [5]:
t0 = time.time()
tg = targets.build_targets(records)
tg.to_csv(config.TARGETS_CSV, index=False)
print(f"targets {tg.shape} in {time.time()-t0:.1f}s   rows/record: "
      f"{tg.groupby('RecordID').size().value_counts().to_dict()}")
tg.groupby("origin_h")[["sofa_total","sofa_now","sofa_delta","deteriorate_24h",
                        "cardio_instability","target_window_empty"]].mean().round(3)


targets (48000, 28) in 4.5s   rows/record: {4: 12000}


,sofa_total,sofa_now,sofa_delta,deteriorate_24h,cardio_instability,target_window_empty
origin_h,,,,,,
24,5.275,5.747,-0.483,0.046,0.684,0.002
30,4.982,5.264,-0.294,0.038,0.653,0.002
36,4.810,4.970,-0.172,0.042,0.627,0.002
42,4.631,4.798,-0.179,0.039,0.603,0.003


In [6]:
# sanity: the T=42 target window IS Day 2 [24,48)h -> must equal notebook SOFA_D2
sc = pd.read_csv(config.SOFA_SCORES_CSV)[["RecordID","SOFA_D2"]]
chk = tg[tg.origin_h == 42].merge(sc, on="RecordID")
d = chk.sofa_total.fillna(0) - chk.SOFA_D2
print(f"T=42 sofa_total vs sofa_scores.SOFA_D2:  r = {stats.pearsonr(chk.sofa_total.fillna(0), chk.SOFA_D2)[0]:.4f}"
      f"   mean|diff| = {d.abs().mean():.4f}   max|diff| = {d.abs().max():.0f}")
assert stats.pearsonr(chk.sofa_total.fillna(0), chk.SOFA_D2)[0] > 0.98

print("\ndelta / deterioration by horizon:")
tg.groupby("origin_h").agg(sofa_delta_mean=("sofa_delta","mean"),
                           sofa_delta_std=("sofa_delta","std"),
                           deteriorate_rate=("deteriorate_24h","mean"),
                           worsen_ge1=("sofa_delta", lambda s: (s>=1).mean())).round(3)


T=42 sofa_total vs sofa_scores.SOFA_D2:  r = 1.0000   mean|diff| = 0.0000   max|diff| = 0

delta / deterioration by horizon:


,sofa_delta_mean,sofa_delta_std,deteriorate_rate,worsen_ge1
origin_h,,,,
24,-0.483,1.595,0.046,0.127
30,-0.294,1.242,0.038,0.132
36,-0.172,1.191,0.042,0.153
42,-0.179,1.228,0.039,0.132


SOFA drifts **down** as the window slides forward (mean delta −0.2 to −0.5) — the
cohort is mostly recovering after day 1. Only ~4 % of (record, horizon) pairs show a
≥ 2-point rise: the deterioration target is rare and will need class weighting.


## 1e  Expanding-window features

Everything for `(RecordID, T)` comes from `icu_records[0, T)` **only**. Families:
`v_<Param>_<stat>` (per-variable summaries), `e_<name>` (engineered hemodynamic /
neuro / renal / respiratory signals), `sofa0_24_*` (rule SOFA on the fixed
admission day), `sofa_now_*` (rule SOFA on `[T−24, T)` = `SOFA_now`), `d_*`
(descriptors). No `outcomes.csv` columns.


In [7]:
t0 = time.time()
ft = features.build_features(records)
ft.to_csv(config.FEATURES_CSV, index=False)
print(f"features {ft.shape}  ({ft.shape[1]-2} columns)  in {time.time()-t0:.0f}s")
fam = pd.Series([c.split('_')[0] for c in ft.columns if c not in ('RecordID','origin_h')]).value_counts()
print("feature families:", fam.to_dict())
nan_frac = ft.drop(columns=['RecordID','origin_h']).isna().mean().sort_values()
print(f"fully-populated columns: {(nan_frac==0).sum()}   median NaN frac: {nan_frac.median():.2f}   "
      f"most-missing: {nan_frac.index[-1]} ({nan_frac.iloc[-1]:.2f})")


features (48000, 409)  (407 columns)  in 130s
feature families: {'v': 360, 'e': 14, 'sofa0': 14, 'sofa': 14, 'd': 5}
fully-populated columns: 60   median NaN frac: 0.18   most-missing: v_Cholesterol_slope12h (1.00)


## 1f  Join & leakage checks


In [8]:
J = ft.merge(tg, on=["RecordID","origin_h"]).merge(sp[["RecordID","split"]], on="RecordID")
print("joined:", J.shape)
print(J.groupby(["split","origin_h"]).size().unstack(), "\n")

# 1) expanding window: any count feature must be non-decreasing in T within a record
ok = True
for v in ["v_HR_count","v_GCS_count","v_Urine_count","v_Creatinine_count"]:
    piv = ft.pivot_table(index="RecordID", columns="origin_h", values=v)
    mono = ((piv[24]<=piv[30]) & (piv[30]<=piv[36]) & (piv[36]<=piv[42])).mean()
    ok &= mono == 1.0
    print(f"  {v:20} monotone non-decreasing in T: {mono:.4f}")
assert ok

# 2) the fixed admission-day SOFA block is identical at every horizon
s = ft.pivot_table(index="RecordID", columns="origin_h", values="sofa0_24_total")
assert (s.nunique(axis=1) == 1).all()
print("  sofa0_24_total identical across all 4 horizons: True")

# 3) no outcome columns
leak = [c for c in ft.columns if c.lower() in
        ("sofa","saps-i","survival","length_of_stay","in-hospital_death")]
assert not leak
print("  outcomes.csv columns in features:", leak or "none")

# 4) every non-empty target row has >=1 organ measured
assert not ((J.target_window_empty == 0) & (J.n_present_target == 0)).any()
print("  non-empty target rows all have >=1 organ scored: True")


joined: (48000, 436)
origin_h    24    30    36    42
split                           
test      1800  1800  1800  1800
train     8400  8400  8400  8400
val       1800  1800  1800  1800 

  v_HR_count           monotone non-decreasing in T: 1.0000
  v_GCS_count          monotone non-decreasing in T: 1.0000
  v_Urine_count        monotone non-decreasing in T: 1.0000
  v_Creatinine_count   monotone non-decreasing in T: 1.0000
  sofa0_24_total identical across all 4 horizons: True
  outcomes.csv columns in features: none
  non-empty target rows all have >=1 organ scored: True


In [9]:
# outcomes.csv reference check (NOT used in the model) — must reproduce the notebook
oc = pd.read_csv(config.OUTCOMES_CSV)[["RecordID","SOFA"]]
ref = ft.loc[ft.origin_h == 24, ["RecordID","sofa0_24_total"]].merge(oc, on="RecordID")
ref = ref[ref.SOFA != -1]
r = stats.pearsonr(ref.sofa0_24_total, ref.SOFA)[0]
bias = (ref.sofa0_24_total - ref.SOFA).mean()
print(f"[0,24)h rule SOFA vs outcomes.SOFA:  r = {r:.3f}   bias = {bias:+.3f}")
print("notebook reported:                   r = 0.933   bias = -0.862")
assert abs(r - 0.933) < 0.01 and abs(bias + 0.862) < 0.03


[0,24)h rule SOFA vs outcomes.SOFA:  r = 0.933   bias = -0.862
notebook reported:                   r = 0.933   bias = -0.862


In [10]:
report = qc.write_qc_report(audit, records, tg, ft, sp)
print(report)


# Phase 1 — data-quality report

## Section-7 implausible-value audit (computed | doc)

| check | computed | doc | ok |
| --- | --- | --- | --- |
| empty_param_rows | 2403 | 2403 | yes |
| empty_param_records | 1594 | 1594 | yes |
| MAP_eq_0 | 86 | 86 | yes |
| PaO2_eq_0 | 3 | 3 | yes |
| SaO2_eq_0 | 2 | 2 | yes |
| Weight_neg1 | 1001 | 1001 | yes |
| Weight_le20 | 353 | 353 | yes |
| Weight_gt300 | 1 | 1 | yes |
| Height_lt100 | 17 | 17 | yes |
| Height_gt250 | 11 | 11 | yes |
| MechVent_rows | 91644 | 91644 | yes |

MechVent distinct values: [1.0] (existence flag only — cardiovascular stays capped 0/1).

## Record duration

- last observation (h): min 0.0, p5 46.3, median 47.5, max 48.0
- records ending before 36 h: 83
- records ending before 42 h: 171
- records ending before 46 h: 406
- records ending before 47 h: 1020

## Target-window coverage (% of 12,000 records with >=1 valid value)

```
 origin_h   window  GCS   HR  Creatinine  Platelets  Bilirubin  PaO2  FiO2  MAP|NIMAP
     

## Phase 1 complete

`modeling/outputs/` now holds `splits.csv`, `records_clean.joblib`, `targets.csv`,
`features.csv`, `qc_report.md` — the joined `(features ⋈ targets ⋈ split)` table is
leak-free and ready to model.

**Next — Phase 2 (EDA + baselines):** target drift per horizon, feature↔target
correlation, and the **persistence baseline** (`SOFA_hat = SOFA_now`) that Phase 3's
models must beat.
